In [ ]:
#use "lib/nb.ml";;

## Drawing Abstract Syntax Trees with GraphViz

In Python, an abstract syntax tree is a *nested tuple*: its leaves are strings and numbers, and an inner node is
a tuple whose first component is the label of the node, followed by the children.  OCaml has no tuples of
variable length.  Therefore, the type `nested` below describes nested tuples: `Str s`, `Int n`, and `Float x` are
the leaves, and `Tuple (label, children)` is an inner node.  The notebooks that use this notebook convert their
abstract syntax trees into this type before they draw them.

In [ ]:
load_lib "graphviz.ml"

In [ ]:
type nested =
  | Str   of string
  | Int   of int
  | Float of float
  | Tuple of string * nested list

`NodeMap`: A dictionary mapping a node's address (a list of ints) to its name (string).  As the Python version
changes this dictionary, we use a *hash table*, which is the mutable dictionary of OCaml.

In [ ]:
type node_map = (int list, string) Hashtbl.t

The function `tuple2dot` takes a nested tuple `t` as its argument.  This nested tuple is interpreted as an
*abstract syntax tree*.   This tree is visualized using the module `Graphviz` from the file `../lib/graphviz.ml`.
The graph is displayed with `Graphviz.display`.

The functions `assign_numbers` and `create_nodes` are defined below.  Since `tuple2dot` calls them, the next
cell uses *forward references*: `!assign_numbers_ref` is the function that is currently stored in the box
`assign_numbers_ref`, and the box is filled once the function has been defined.

In [ ]:
let not_yet_defined _ = failwith "not yet defined"

let assign_numbers_ref : (int list -> nested -> node_map -> int -> int) ref = ref not_yet_defined
let create_nodes_ref   : (Graphviz.t -> int list -> nested -> node_map -> unit) ref = ref not_yet_defined

let tuple2dot (t : nested) : Graphviz.t =
  let dot = Graphviz.digraph "Abstract Syntax Tree" in
  let nodes_2_names : node_map = Hashtbl.create 16 in
  ignore (!assign_numbers_ref [] t nodes_2_names 0);
  !create_nodes_ref dot [] t nodes_2_names;
  dot

The function `assign_numbers` takes four arguments:
- `address` is the address of the node `t`, i.e. the list of the positions of the children that lead from the root to `t`,
- `t` is a nested tuple that is interpreted as a tree,
- `nodes2Numbers` is a hash table,
- `n` is a natural number.

Given a tree `t` that is represented as a nested tuple, the function `assign_numbers` assigns a unique natural number 
to every node of `t`.  This assignment is stored in the hash table `nodes2Numbers`. `n` is the first natural number
that is used.  The function returns the smallest natural number that is still unused.

In [ ]:
let rec assign_numbers (address : int list) (t : nested) (nodes2Numbers : node_map) (n : int) : int =
  Hashtbl.replace nodes2Numbers address (string_of_int n);
  match t with
  | Str _ | Int _ | Float _ -> n + 1
  | Tuple (_, children) ->
      let n = ref (n + 1) in
      List.iteri (fun j child -> n := assign_numbers (address @ [ j + 1 ]) child nodes2Numbers !n) children;
      !n
;;
assign_numbers_ref := assign_numbers

The function `create_nodes` takes four arguments:
- `dot` is a graph of type `Graphviz.t`,
- `a` is the address of the node `t`,
- `t` is an abstract syntax tree represented as a nested tuple.
- `nodes_2_names` is a hash table mapping nodes in `t` to unique names
   that can be used as node names in graphviz.
   
The function creates the nodes in `t` and connects them via directed edges so that `t` is represented as a tree.
The function `float_repr` writes a floating point number like Python does.

In [ ]:
let rec create_nodes (dot : Graphviz.t) (a : int list) (t : nested) (nodes_2_names : node_map) : unit =
  let root = Hashtbl.find nodes_2_names a in
  match t with
  | Str s -> Graphviz.node dot ~attrs:[ ("label", s) ] root
  | Int n -> Graphviz.node dot ~attrs:[ ("label", string_of_int n) ] root
  | Float x -> Graphviz.node dot ~attrs:[ ("label", float_repr x) ] root
  | Tuple (label, children) ->
      Graphviz.node dot ~attrs:[ ("label", label) ] root;
      List.iteri
        (fun j c ->
          let child = Hashtbl.find nodes_2_names (a @ [ j + 1 ]) in
          Graphviz.edge dot root child;
          create_nodes dot (a @ [ j + 1 ]) c nodes_2_names)
        children
;;
create_nodes_ref := create_nodes

Finally, the function `nested_to_string` writes a nested tuple in the syntax of Python, e.g. `('+', 1, 'x')`.
This is the format that Python's function `print` uses for nested tuples.

In [ ]:
let rec nested_to_string (t : nested) : string =
  let py_str s = if String.contains s '\'' && not (String.contains s '"') then "\"" ^ s ^ "\"" else "'" ^ s ^ "'" in
  match t with
  | Str s -> py_str s
  | Int n -> string_of_int n
  | Float x -> float_repr x
  | Tuple (label, []) -> "(" ^ py_str label ^ ",)"
  | Tuple (label, children) -> "(" ^ String.concat ", " (py_str label :: List.map nested_to_string children) ^ ")"